# TCC · EDA de Marketing Mix Modeling com Google Meridian

**Dados simulados oficiais, foco GEO × TEMPO.** Não há estimação posterior, ROI ou recomendação de orçamento.

Execute de cima para baixo em runtime novo Python 3.12+ (CPU). A primeira execução instala dependências; isso pode levar alguns minutos. Se o Colab já tinha versões diferentes de NumPy/TensorFlow carregadas, reinicie a sessão após instalar e execute novamente.

O bootstrap clona o repositório público quando necessário, preserva instalações compatíveis e salva tudo na pasta do projeto. A fonte de dados e a API estão fixadas por commit/checksum. Para uma cópia histórica exata do projeto, configure `REPO_REF` com o commit da entrega. Os dados não possuem nomes de canais reais ou localização geográfica real.


In [1]:
# Execute esta célula primeiro. Em Colab novo, usa somente CPU.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

REPO_URL = "https://github.com/thiagossilva17/MMM_Google_Meridian_Project.git"
REPO_REF = "main"  # Para repetir uma execução publicada, substitua pelo SHA do commit.
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
cwd = Path.cwd()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src/config.py").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd / "MMM_Google_Meridian_Project"
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_REF], check=True)
    elif not (PROJECT_ROOT / "src/config.py").exists():
        raise RuntimeError("Diretório de destino existe, mas não contém o projeto esperado.")

required = ["meridian", "pandas", "numpy", "scipy", "matplotlib", "statsmodels", "tabulate"]
missing = any(importlib.util.find_spec(package) is None for package in required)
try:
    wrong_meridian = importlib.metadata.version("google-meridian") != "2.1.0"
except importlib.metadata.PackageNotFoundError:
    wrong_meridian = True
if missing or wrong_meridian:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "requirements-colab.txt")], check=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs/.mplcache"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from src.config import SEED, ROOT
from src.data import load_panel
from src.reporting import show
from src.provenance import verify_checkout, verify_meridian_origin
print("Código:",verify_checkout(PROJECT_ROOT, REPO_REF))
print("Meridian:",verify_meridian_origin())
import numpy as np
np.random.seed(SEED)
geo_time_panel = load_panel()
print(f"Colab: {IN_COLAB}; projeto: {ROOT}; painel: {geo_time_panel.shape}")


Código: {'requested_ref': 'main', 'resolved_commit': 'ac9960a3a86e7af07a507121043c6914fcbc4195', 'dirty': True}
Meridian: {'url': 'https://github.com/google/meridian.git', 'vcs_info': {'commit_id': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'requested_revision': '02111531f8661373aa7b6ba31c316c67d72d1dd2', 'vcs': 'git'}}
Colab: False; projeto: /workspace/scratch/5d823022ed95/tcc-revision; painel: (6240, 19)


O manifesto verifica código, raw, ambiente, conclusão das etapas e hashes dos outputs; resultados incompatíveis são recalculados.

In [2]:
from src.provenance import require_stages
try:
    require_stages(range(5))
except RuntimeError as stale:
    print('Recalculando etapas anteriores:',stale)
    from src.eda import data_audit,general,temporal,relationships
    from src.geo_analysis import geo_analysis,media_geo
    from src.official_eda import official_eda
    for analysis in [data_audit,general,temporal,geo_analysis,media_geo,relationships,official_eda][:5]:
        analysis(geo_time_panel)


## 05 — Quais sinais se movem juntos ou podem ser redundantes?

### Pergunta
Quais sinais se movem juntos ou podem ser redundantes?

### Motivação
Mídia pode responder à demanda e compartilhar tendência, sazonalidade ou alocação com outros canais. Isso limita interpretações ingênuas.

### Método
Pearson/Spearman nacional, overall, within e two-way; matrizes separadas de gasto e mídia; VIF com intercepto em preditores, sem duplicar spend e impressões; lags de 0 a 8 semanas, com contagem das observações alinhadas.

### Interpretação antes de olhar os resultados
Não usar VIF como regra automática de remoção. Correlações entre gasto e impressões podem ser contábeis. Lags não estimam adstock. Não calcular significância supondo independência de linhas geo-temporais.

**Dependências:** painel validado e funções em `src/eda.py`. O código reutilizável está comentado e pode ser aberto para inspecionar as fórmulas. Todos os resultados são calculados na execução, não preenchidos manualmente.


In [3]:
from src.eda import relationships
report_05 = relationships(geo_time_panel)
show(report_05,part="summary")

### Resultado observado e conclusão parcial

Exportamos Pearson e Spearman em quatro níveis: nacional, painel bruto, within-geo e residual two-way. Within remove a média de cada geo; two-way remove também a média semanal comum. Spearman de resíduos mede a ordenação dos resíduos e não é uma correlação parcial de postos.

O maior VIF within entre preditores é 2.11. A regressão auxiliar inclui intercepto e colunas padronizadas. Spend e impressões são diagnosticados em matrizes separadas; colocá-los juntos no VIF duplicaria sinais de custo praticamente fixo. Constantes são indefinidas e dependência linear perfeita gera infinito.

A maior associação absoluta within entre preditores distintos é Channel4_impression × competitor_sales_control (0.550); o ranking exclui pares gasto–impressões do mesmo canal. Lags 0–8 são exploratórios, sem selecionar adstock. Não calculamos p-valores que presumiriam independência das 6,240 linhas: existe dependência temporal e geográfica.

Diferenças entre correlações nacionais e within podem resultar de composição populacional, tendência, sazonalidade e mídia alocada em antecipação à demanda. Correlação mídia–KPI não constitui efeito de mídia.

**Channel2 × Channel3:** nacional 0.7003, within 0.5035, two-way 0.4511. Parte do movimento conjunto acompanha composição/tempo comum, mas há associação residual. O ranking exclui gasto–impressões do mesmo canal.

competitor_sales_control: R² tempo 0.332; VIF within 1.83; maior associação two-way com Channel4_impression (+0.478). Justificar papel pré-tratamento/confundidor no DAG; associação não autoriza inclusão causal.

sentiment_score_control: R² tempo 0.274; VIF within 1.88; maior associação two-way com Organic_channel0_impression (+0.437). Justificar papel pré-tratamento/confundidor no DAG; associação não autoriza inclusão causal.

Promo: R² tempo 0.198; VIF within 1.28; maior associação two-way com Channel0_impression (+0.297). Justificar papel pré-tratamento/confundidor no DAG; associação não autoriza inclusão causal.

### O que ainda NÃO podemos concluir

As relações são descritivas. Não estimamos ROI, contribuição incremental ou efeitos causais.
Os dados são simulados: padrões não descrevem um mercado real e não validam, por si, a identificação de um MMM.

### CHECKPOINT

- **O que descobrimos?** **Channel2 × Channel3:** nacional 0.7003, within 0.5035, two-way 0.4511. Parte do movimento conjunto acompanha composição/tempo comum, mas há associação residual. O ranking exclui gasto–impressões do mesmo canal.
- **Quais problemas apareceram?** VIF moderado não resolve confundimento nem colinearidade após adstock/saturação.
- **O que falta investigar?** Checagens independentes da ferramenta oficial e comparação de definições.
- **Podemos avançar?** Podemos prosseguir com a investigação exploratória após revisar estes achados; isso não aprova automaticamente a modelagem.


### Evidências e leitura das figuras
Tabelas pequenas são exibidas integralmente. As sínteses selecionam os casos relevantes; os CSVs mantêm os dados completos.

In [4]:
show(report_05,part="evidence")

**vif** — 36 linhas; CSV completo em `outputs/tables/`.

,variable,level,vif,status
0,Channel0_impression,national,1.719037,finite
1,Channel1_impression,national,1.506175,finite
2,Channel2_impression,national,2.890948,finite
3,Channel3_impression,national,3.857286,finite
4,Channel4_impression,national,2.563407,finite
5,Organic_channel0_impression,national,2.376850,finite
6,competitor_sales_control,national,3.793200,finite
7,sentiment_score_control,national,3.461720,finite
8,Promo,national,1.782877,finite
9,Channel0_impression,overall,1.632356,finite


**correlations** — 15 linhas; CSV completo em `outputs/tables/`.

,level,method,left,right,correlation
486,within,pearson,Channel4_impression,competitor_sales_control,0.550356
478,within,pearson,Channel3_impression,sentiment_score_control,0.506064
459,within,pearson,Channel2_impression,Channel3_impression,0.503550
477,within,pearson,Channel3_impression,competitor_sales_control,0.498532
470,within,pearson,Channel3_impression,Channel4_impression,0.479959
520,within,pearson,Organic_channel0_impression,sentiment_score_control,0.476936
476,within,pearson,Channel3_impression,Organic_channel0_impression,0.454943
467,within,pearson,Channel2_impression,competitor_sales_control,0.438038
460,within,pearson,Channel2_impression,Channel4_impression,0.435983
468,within,pearson,Channel2_impression,sentiment_score_control,0.402763


**lags** — 45 linhas; CSV completo em `outputs/tables/`.

...

[Figura gerada; veja outputs/figures ou execute a célula.]

**Como ler — vif:** Compare com os resultados numéricos e a interpretação do capítulo. Unidade e denominador delimitam a leitura; associação não demonstra eficácia.

[Figura gerada; veja outputs/figures ou execute a célula.]

**Como ler — lags:** Compare com os resultados numéricos e a interpretação do capítulo. Unidade e denominador delimitam a leitura; associação não demonstra eficácia.

[Figura gerada; veja outputs/figures ou execute a célula.]